# PDF Retrieval And Provenance

Follow two included PDFs from text extraction to an evidence trace. The goal is to inspect what retrieval actually found before using it in a structured workflow.


## Extract the sample PDFs

The included `pdfminer` loader runs offline. It is a heuristic parser, so inspect its section titles and text before treating a result as evidence.


In [ ]:
import sys
from pathlib import Path

if (Path.cwd() / "notebooks" / "epilens_nb.py").is_file():
    sys.path.insert(0, str(Path.cwd() / "notebooks"))
import epilens_nb as nb

WORK_DIR = nb.bootstrap()

from textwrap import shorten

pdf_paths = sorted(nb.pdf_samples_dir().glob("*.pdf"))
if not pdf_paths:
    raise FileNotFoundError("No sample PDFs found in notebooks/pdf_samples/.")

[path.name for path in pdf_paths]


In [ ]:
from epilens.db.in_memory_academic_db import InMemoryAcademicDB
from epilens.rag.ingestion.document_loader import DocumentLoaderFactory

strategy_name = "pdf-samples-pdfminer"
loader = DocumentLoaderFactory.get_loader("pdfminer")
academic_db = InMemoryAcademicDB(backup_file=str(WORK_DIR / "academic_db.json"))

for pdf_path in pdf_paths:
    sections, metadata, references = loader.load(pdf_path)
    metadata.file_path = str(pdf_path)
    doc_id = pdf_path.stem

    academic_db.insert(doc_id, "sections", strategy_name, [section.to_dict() for section in sections])
    academic_db.insert(doc_id, "metadata", strategy_name, metadata.to_dict())
    academic_db.insert(doc_id, "references", strategy_name, [ref.to_dict() for ref in references])

    print(f"{doc_id}: {len(sections)} sections, {len(references)} parsed references, title={metadata.title!r}")

doc_ids = academic_db.list_docs(strategy_name)


## Index the extracted text

The tiny keyword embedder keeps this example offline. Its scores reflect keyword overlap, not semantic understanding; a relevant chunk can still rank below a table or reference entry.


In [ ]:
embedder = nb.TinyKeywordEmbedder()
embedder.model_name, embedder.dim

In [ ]:
from epilens.rag.indexing.chunking import FixedSizeChunker
from epilens.rag.indexing.indexer import Indexer
from epilens.rag.retrieval.candidates import SemanticCandidateRetriever
from epilens.rag.retrieval.retriever import Retriever
from epilens.vectordb.file import FileDB

vdb = FileDB(str(WORK_DIR / "file_index"))
indexer = Indexer(
    vdb,
    embedder=embedder,
    chunker=FixedSizeChunker(chunk_size=1200, chunk_overlap=150),
)

for doc_id in doc_ids:
    sections = academic_db.retrieve(doc_id, "sections", strategy_name) or []
    metadata = academic_db.retrieve(doc_id, "metadata", strategy_name)
    indexer.index_paper(sections, metadata, paper_id=doc_id)

vdb.save()
semantic_candidates = SemanticCandidateRetriever(vdb, dense_embedder=embedder)
retriever = Retriever(vdb, candidate_retrievers=[semantic_candidates], use_rerank=False)

len(vdb.get_points()), vdb.get_embedding_model(), vdb.get_chunking_config()


## Search across both PDFs

This query targets the avian influenza paper's clinical management discussion. Read the text and section title, not just the similarity score.


In [ ]:
query = "influenza patient treatment mortality"
results = retriever.retrieve(query, top_k=3)

for rank, result in enumerate(results, start=1):
    preview = shorten(result.text.replace("\n", " "), width=320)
    print(f"{rank}. {result.paper_id} | {result.section_title} | score={result.similarity_score:.3f}")
    print(f"   {preview}\n")


## Narrow the search to one PDF

Now ask where the Daegu COVID-19 study obtained its case data. Paper-scoped search prevents passages from the other PDF entering this evidence set.


In [ ]:
paper_id = "2021_Kim_Jong-Yeon"
paper_query = "source data survey"
paper_results = retriever.retrieve_by_paper(paper_query, paper_id, top_k=3)

for rank, result in enumerate(paper_results, start=1):
    preview = shorten(result.text.replace("\n", " "), width=320)
    print(f"{rank}. {result.section_title} | score={result.similarity_score:.3f}")
    print(f"   {preview}\n")


## Keep a provenance trace

In this sample, the first paper-scoped hit describes the surveys used for case data. `NoLLMGenerator` copies retrieved text into an answer and records its source; it does not synthesize or verify a claim.


In [ ]:
from epilens.rag.generation.nollm_generator import NoLLMGenerator

source = paper_results[0]
trace = NoLLMGenerator().generate([source], question=paper_query)
print("Retrieved text:")
print(shorten(trace.answer.replace("\n", " "), width=400))

print(f"\nParsed section title: {source.section_title}")
print("Provenance:")
for evidence in trace.evidences:
    print(f"- {evidence.paper_id}: {shorten(evidence.snippet, width=160)}")


`pdfminer` can mistake a reference or table row for a section, and may use the file stem as the title or miss references altogether. The trace tells you where a passage came from, but a person still needs to check the PDF and the passage's meaning. Notebook 05 uses clean, controlled sections to introduce structured extraction.
